# Digital Support Teams Signup Clicks

This notebook tracks traffic to the Digital Support page and outbound clicks from that page to Microsoft Teams event links.

It groups clicks by destination domain rather than individual URLs, so rotating event-session links still roll up into one total.

### Quick start
1. Run the **Setup (run once)** cell.
2. In **Parameters**, set the date window if needed.
3. Run the query cell and confirm there are `click` events for the target domain.
4. Review the summary and daily trend charts.

**Metrics:** page views, page-view sessions, outbound click events, clicked sessions, click-through rate  
**Data source:** GA4 BigQuery export (`events_*`)  
**Important caveat:** this works only if GA4 outbound click tracking is present in the export as `click` events.


In [ ]:
#@title Setup (run once)
import sys
import os

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    if not os.path.exists("lla-data"):
        !git clone -q https://github.com/aidoanto/lla-data.git
    repo = os.path.abspath("lla-data")
    if repo not in sys.path:
        sys.path.insert(0, repo)
    !pip install -U -q 'plotly>=6.1.1' 'kaleido>=1.2.0'
else:
    for p in ("..", "../.."):
        ap = os.path.abspath(p)
        if ap not in sys.path:
            sys.path.insert(0, ap)

import pandas as pd
import plotly.express as px
from google.cloud import bigquery

import lifeline_theme
from lla_data import config
from lla_data.bq import (
    build_date_params,
    default_query_window,
    dry_run_bytes,
    get_client,
    load_sql_template,
    run_query,
)

lifeline_theme.inject_fonts()

client = get_client()


In [ ]:
#@title Parameters
DAYS_BACK = 92 #@param {type:"integer"}
SOURCE_PAGE = "https://www.lifeline.org.au/get-involved/volunteer/volunteer-as-a-crisis-supporter/digital-support" #@param {type:"string"}
OUTBOUND_DOMAIN = "events.teams.microsoft.com" #@param {type:"string"}

window = default_query_window(days_back=DAYS_BACK)
analysis_start = window.start_date
analysis_end = window.end_date
analysis_window_label = f"{analysis_start:%Y-%m-%d} to {analysis_end:%Y-%m-%d}"

print(f"Project: {config.PROJECT_ID}")
print(f"GA4 dataset: {config.GA4_DATASET}")
print(f"Window: {analysis_start} to {analysis_end}")
print(f"Source page: {SOURCE_PAGE}")
print(f"Outbound domain: {OUTBOUND_DOMAIN}")


In [ ]:
query = load_sql_template(
    "sql/notebooks/ga4_outbound_domain_funnel.sql",
    project_id=config.PROJECT_ID,
    ga4_dataset=config.GA4_DATASET,
)

params = build_date_params(window) + [
    bigquery.ScalarQueryParameter("source_page", "STRING", SOURCE_PAGE),
    bigquery.ScalarQueryParameter("outbound_domain", "STRING", OUTBOUND_DOMAIN),
]

estimated_bytes = dry_run_bytes(client, query, params=params)
print(f"Estimated bytes scanned: {estimated_bytes:,}")

df = run_query(client, query, params=params)
df.head()


In [ ]:
totals = {
    "Page view events": int(df["page_view_events"].sum()),
    "Page-view sessions": int(df["page_view_sessions"].sum()),
    "Teams click events": int(df["outbound_click_events"].sum()),
    "Sessions with Teams click": int(df["outbound_click_sessions"].sum()),
    "Event click-through rate": df["outbound_click_events"].sum() / df["page_view_events"].sum() if df["page_view_events"].sum() else 0,
    "Session click-through rate": df["outbound_click_sessions"].sum() / df["page_view_sessions"].sum() if df["page_view_sessions"].sum() else 0,
}

summary = pd.DataFrame(
    {
        "metric": list(totals.keys()),
        "value": list(totals.values()),
    }
)

summary["formatted_value"] = summary.apply(
    lambda row: f"{row['value']:.1%}" if "rate" in row["metric"].lower() else f"{int(row['value']):,}",
    axis=1,
)

summary[["metric", "formatted_value"]]


In [ ]:
trend_df = df.melt(
    id_vars="report_date",
    value_vars=["page_view_events", "outbound_click_events"],
    var_name="metric",
    value_name="count",
)

trend_df["metric"] = trend_df["metric"].map(
    {
        "page_view_events": "Page view events",
        "outbound_click_events": "Teams click events",
    }
)

fig = px.line(
    trend_df,
    x="report_date",
    y="count",
    color="metric",
    template="lifeline",
    title=f"Digital Support Page Views vs Teams Clicks ({analysis_window_label})",
    labels={"report_date": "Date", "count": "Events", "metric": "Metric"},
)
fig.update_layout(hovermode="x unified")
lifeline_theme.add_lifeline_logo(fig)
fig.show()


In [ ]:
fig = px.line(
    df,
    x="report_date",
    y="session_click_through_rate",
    template="lifeline",
    title=f"Digital Support Session Click-Through Rate ({analysis_window_label})",
    labels={
        "report_date": "Date",
        "session_click_through_rate": "Session CTR",
    },
)
fig.update_traces(line=dict(width=3))
fig.update_yaxes(tickformat=".0%")
fig.update_layout(hovermode="x unified")
lifeline_theme.add_lifeline_logo(fig)
fig.show()


## Interpretation notes

- `Teams click events` counts all outbound clicks from this page to `events.teams.microsoft.com`, regardless of which specific event-session URL was live at the time.
- `Sessions with Teams click` is the cleaner funnel measure if one user can click more than once in a session.
- If this notebook returns page views but zero clicks for all time, check whether GA4 enhanced measurement outbound clicks were enabled when the page was live.
